<div align="center">

# Patra Toolkit: Model Cards & Datasheets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-to-Insight-Center/patra-toolkit/blob/main/examples/notebooks/ModelCardAndDatasheetDemo.ipynb)

</div>

[Patra](https://github.com/Data-to-Insight-Center/patra-knowledge-base) documents AI/ML models and the datasets that trained them as structured, machine-actionable metadata -- **Model Cards** and **Datasheets** -- instead of a README that goes stale. This notebook builds a Model Card and Datasheet, submits them to a Patra server, looks up existing records, and (optionally) streams a live inference run through CKN.

## Install

In [1]:
!pip install -q patra-toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00


## Connect to a Patra server

Reads (list/get) work anonymously. Writes (`submit`) need a Tapis token -- get one with `mc.authenticate(username=..., password=...)`.

In [2]:
patra_server_url = "https://patrabackenddemo.pods.icicleai.tapis.io/"
tapis_token = None

## Build a Model Card

Only `name` is required -- everything else, including the attached `AIModel`, is optional.

In [3]:
from patra_toolkit import ModelCard, AIModel

mc = ModelCard(
    name="UCI_Adult_Model",
    version="1.0",
    short_description="UCI Adult Data analysis using Tensorflow for demonstration of Patra Model Cards.",
    full_description="We have trained a ML model using the tensorflow framework to predict income for the UCI Adult Dataset. We leverage this data to run the Patra model cards to capture metadata about the model as well as fairness and explainability metrics.",
    keywords="uci adult, tensorflow, explainability, fairness, patra",
    author="0009-0009-9817-7042",
    input_type="Tabular",
    category="classification",
    foundational_model="None",
    citation="Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI.",
)
mc.input_data = "https://archive.ics.uci.edu/dataset/2/adult"
mc.output_data = "https://huggingface.co/patra-iu/neelk-uci_adult_model-1.0"

mc.ai_model = AIModel(
    name="Random Forest",
    version="0.1",
    description="Census classification problem using Random Forest",
    owner="neelk",
    location="https://github.iu.edu/swithana/mcwork/randomforest/adult_model.pkl",
    license="BSD-3 Clause",
    framework="sklearn",
    model_type="random_forest",
    test_accuracy=0.85,
)

mc.validate()

True

## Build a Datasheet

Documents the dataset the model was trained on, the same way.

In [4]:
from patra_toolkit import Datasheet

ds = Datasheet(publication_year=2025, version="1.0")
ds.add_title("UCI Adult Dataset")
ds.add_creator("Becker, B.")
ds.add_creator("Kohavi, R.")
ds.add_description("Predict whether income exceeds $50K/yr based on census data.", "Abstract")

ds.validate()

True

## Submit

Raises `PatraModelExistsError`/`PatraDatasheetExistsError` if an equivalent record already exists -- bump `version` to submit a new one.

In [5]:
if tapis_token:
    mc.submit(patra_server_url=patra_server_url, token=tapis_token)
    ds.submit(patra_server_url=patra_server_url, token=tapis_token)
    print(f"Model Card: {mc.uuid}\nDatasheet: {ds.uuid}")
else:
    print("No token set -- call mc.authenticate(username=..., password=...) to get one and actually submit.")

No token set -- call mc.authenticate(username=..., password=...) to get one and actually submit.


## Discover existing records

`list_*` also takes `q=` (substring search) and `skip=`/`limit=` for paging.

In [6]:
import pandas as pd

model_cards = ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(model_cards)

,uuid,name,categories,author,version,short_description,is_gated,is_private,updated_at
0,8c517ed0-c9c0-4f57-bb9d-f066ab4ec34e,BioCLIP 2 (via pybioclip),classification,John Bradley / Imageomics Institute,v2.0,Biology foundation model for taxonomic classif...,False,False,2026-05-22T19:50:39.783085+00:00
1,b404980f-438a-4760-b358-f6325e6c8f2d,Crop Weed YOLO Model CNW,object detection,Tommy,v1,Custom YOLO model for crop and weed detection.,False,False,2026-05-04T23:18:53.064965+00:00
2,b521f10c-6845-44cd-9021-5bf407472633,Deeplab_V3 Engine,Segmentation,Harikesh Byrandurga Gopinath,1.0.0,TensorRT optimized Deeplab_V3 engine for segme...,False,False,2026-06-11T20:41:26.003210+00:00
3,31b191a5-121c-4b5f-8266-f99da0f6580f,GoogLeNet for Image Classification,classification,swithana,1.0,Image classification using GoogLeNet.,False,False,2026-05-04T23:18:53.064965+00:00
4,62cb5c70-08fc-4899-9522-483274e21ef2,Grounding DINO — grounded open-vocabulary dete...,computer vision,haeparth,1.0,Grounding-oriented detector aligning language ...,False,False,2026-05-05T05:14:47.194483+00:00


In [7]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=model_cards[0]["uuid"], token=tapis_token)

{'uuid': '8c517ed0-c9c0-4f57-bb9d-f066ab4ec34e',
 'name': 'BioCLIP 2 (via pybioclip)',
 'version': 'v2.0',
 'short_description': 'Biology foundation model for taxonomic classification and trait prediction.',
 'full_description': 'BioCLIP 2 is a large-scale vision-language foundation model for biological understanding. It is trained on the TreeOfLife-200M dataset, which contains 214 million images across 952k taxa. Unlike standard CLIP, it utilizes hierarchical contrastive learning to align image representations with the taxonomic tree of life, allowing it to generalize to unseen species and even predict ecological traits like habitat and life stage.',
 'keywords': 'biology, taxonomy, wildlife, organism detection, zero-shot, pybioclip, tree of life',
 'author': 'John Bradley / Imageomics Institute',
 'input_data': 'https://huggingface.co/datasets/imageomics/TreeOfLife-200M',
 'output_data': 'https://github.com/Imageomics/pybioclip',
 'input_type': 'images',
 'categories': 'classificatio

In [8]:
datasheets = Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(datasheets)

,uuid,title,creator,category,is_private,updated_at
0,56688632-9f6d-48da-80d6-76f8a711be19,Big Bird,Swathi V,None,False,2026-07-27T20:59:48.646858+00:00
1,d169e2ed-2435-49f0-93ef-207abcc44ede,CKN Inference Demo Images,Demo Author,None,False,2026-07-28T01:09:13.107950+00:00
2,9c132373-f663-426d-b5d6-1de5811db3fe,COCO 2017 Train,Swathi V,`object-detection,False,2026-07-27T19:51:14.318935+00:00
3,2a7b541d-d3d4-4969-8639-576830ad3d95,Continually Adapt or Not (CAN) Benchmark,ICICLE AI Institute,Camera trap,False,2026-06-03T22:28:37.608303+00:00
4,ab55bd2e-5146-4509-b862-cc0292626456,HLO Feature Dataset for Deep Learning Resource...,ICICLE AI Institute,Graph machine learning,False,2026-06-03T22:28:37.608303+00:00


In [9]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=datasheets[0]["uuid"], token=tapis_token)

{'uuid': '56688632-9f6d-48da-80d6-76f8a711be19',
 'publication_year': None,
 'resource_type': 'Dataset',
 'resource_type_general': None,
 'size': None,
 'format': None,
 'version': '1.0.0',
 'is_private': False,
 'updated_at': '2026-07-27T20:59:48.646858+00:00',
 'creators': [{'creator_name': 'Swathi V',
   'name_type': None,
   'lang': None,
   'given_name': None,
   'family_name': None,
   'name_identifier': None,
   'name_identifier_scheme': None,
   'name_id_scheme_uri': None,
   'affiliation': None,
   'affiliation_identifier': None,
   'affiliation_identifier_scheme': None,
   'affiliation_scheme_uri': None}],
 'titles': [{'title': 'Big Bird', 'title_type': None, 'lang': None}],
 'publisher': {'name': 'LILA BC',
  'publisher_identifier': None,
  'publisher_identifier_scheme': None,
  'scheme_uri': None,
  'lang': None},
 'subjects': [],
 'contributors': [],
 'dates': [],
 'alternate_identifiers': [],
 'related_identifiers': [{'related_identifier': 'https://awscli.amazonaws.com/v2

## Run a live inference experiment (optional)

`run_experiment()` fetches a Model Card + Datasheet, downloads the model and sample images they reference, runs inference, and streams a metric event per image to CKN (Kafka) so results show up in Patra's web app live. This uses different, pre-existing demo records with real downloadable weights/images -- not the ones built above.

In [10]:
!pip install -q "patra-toolkit[experiments]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.3 MB/s eta 0:00:00


In [11]:
import torchvision
from patra_toolkit import run_experiment

model_card_uuid = "56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6"  # pretrained ResNet50
datasheet_uuid = "d169e2ed-2435-49f0-93ef-207abcc44ede"  # Lorem Picsum sample images
categories = torchvision.models.ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

Registering is idempotent -- safe to re-run.

In [14]:
import requests

user_id = "neelk"  # replace with your own

requests.post(f"{patra_server_url.rstrip('/')}/users", json={"username": user_id}).raise_for_status()

In [15]:
result = run_experiment(
    model_card_uuid=model_card_uuid,
    datasheet_uuid=datasheet_uuid,
    patra_server_url=patra_server_url,
    ckn_broker_url="cknbroker.pods.icicleai.tapis.io:443",  # your broker's advertised external address
    user_id=user_id,
    domain="animal-ecology",
    token=tapis_token,
    categories=categories,
)
result

{'experiment_id': 'experiment-eb0f8604',
 'model_card_uuid': '56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6',
 'datasheet_uuid': 'd169e2ed-2435-49f0-93ef-207abcc44ede',
 'user_id': 'neelk',
 'device_id': 'demo-edge-device',
 'domain': 'animal-ecology',
 'num_events_produced': 20,
 'events': [{'domain': 'animal-ecology',
   'device_id': 'demo-edge-device',
   'experiment_id': 'experiment-eb0f8604',
   'user_id': 'neelk',
   'model_id': '56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6',
   'UUID': 'bcf39fc6-7987-4f51-a66f-14681a1efca7',
   'image_name': '0.jpg',
   'ground_truth': None,
   'image_count': 1,
   'image_receiving_timestamp': '2026-07-28T16:24:22Z',
   'image_scoring_timestamp': '2026-07-28T16:24:22Z',
   'image_store_delete_time': '2026-07-28T16:24:22Z',
   'image_decision': 'Save',
   'label': 'notebook',
   'probability': 0.6385801,
   'flattened_scores': '[{"label": "notebook", "probability": 0.6386}, {"label": "laptop", "probability": 0.142}, {"label": "space bar", "probability": 0.0681}, 

View results at `https://patrabackend.pods.icicleai.tapis.io/experiments/animal-ecology/users/demo_user/summary` -- the server CKN's sink connector actually writes to -- or in the Patra web app at **[patra.pods.icicleai.tapis.io](https://patra.pods.icicleai.tapis.io)** or **[patrademo.pods.icicleai.tapis.io](https://patrademo.pods.icicleai.tapis.io)**, under **Experiments → Animal Ecology**. CKN streams into both `patradb` and `patradb_demo`, so either works.

## Next steps

- **Fairness & explainability**: `mc.populate_bias(...)` ([fairlearn](https://fairlearn.org/)) and `mc.populate_xai(...)` ([SHAP](https://shap.readthedocs.io/)).
- **Full field reference**: [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md).
- **More examples**: [examples/notebooks/](https://github.com/Data-to-Insight-Center/patra-toolkit/tree/main/examples/notebooks).
- **Browse visually**: the Patra web app (`patra-frontend/` in this workspace).